# CrashGuard Milestone 3.1 — tracked-motion temporal crash classifier
This notebook trains on real CCD footage only after Drive authorization and explicit terms acceptance. Synthetic smoke-test metrics are not used here. YOLO remains the object detector; the GRU supplies conservative temporal evidence. Run cells in order.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil, subprocess, sys, os, json, time
import torch
HAS_GPU = torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0) if HAS_GPU else 'not assigned — CPU fallback works but full training will be slow')
if HAS_GPU:
    print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print('Runtime free GB:', round(shutil.disk_usage('/content').free / 2**30, 2))
print('Drive free GB:', round(shutil.disk_usage('/content/drive').free / 2**30, 2))

## Project and dependencies
Copy the CrashGuard project folder to My Drive once. The next cell locates it automatically, so paths do not need hand repair.

In [ ]:
matches = [p.parent for p in Path('/content/drive/MyDrive').rglob('manage.py') if (p.parent / 'ml/training/train_temporal.py').is_file()]
if not matches:
    import zipfile
    bundles = list(Path('/content/drive/MyDrive').rglob('CrashGuard_Colab_Training_Bundle.zip'))
    assert len(bundles) == 1, 'Upload CrashGuard_Colab_Training_Bundle.zip to My Drive, then rerun this cell.'
    target = Path('/content/drive/MyDrive/CrashGuardTraining')
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundles[0]) as archive: archive.extractall(target)
    matches = [target]
PROJECT_ROOT = matches[0]
print('Project:', PROJECT_ROOT)
%cd $PROJECT_ROOT
%pip install -q -r ml/requirements-colab.txt
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'crash_monitor.settings')
subprocess.run([sys.executable, 'manage.py', 'check'], check=True)

## Dataset authorization and acquisition
CCD documents 1,500 crash clips and 3,000 normal BDD100K clips. The repository code is MIT, but that does not automatically grant redistribution rights for the underlying YouTube/BDD100K footage. Review the official CCD and BDD100K terms before changing the flag. Authenticated or quota-limited downloads may require opening the printed CCD link in the same Google account. Data stays in Drive, not on the laptop.

In [ ]:
ACCEPT_CCD_DATASET_TERMS = False  # change only after reviewing the source terms
CCD_FOLDER_URL = 'https://drive.google.com/drive/folders/1NUwC-bkka0-iPqhEhIgsXWtj0DA2MR-F'
CCD_FILE_IDS = {'Crash-1500.txt': '13OgrD0-8cKG0X00MlA6JXr0G_JJmHYXg', 'Crash-1500.zip': '1fmcwGhr8JT9YfLUrlcvuCZ3ychi2eyFP', 'Normal.zip': '11ErpWQmmV5au2JOQVxwYl3ebtuugdlan'}
EXPECTED_BYTES = {'Crash-1500.txt': 276792, 'Crash-1500.zip': 793091291, 'Normal.zip': 6445547827}
DATA_ROOT = Path('/content/drive/MyDrive/CrashGuardData')
CCD_ROOT = DATA_ROOT / 'CarCrash'
VIDEO_ROOT = CCD_ROOT / 'videos'
VIDEO_ROOT.mkdir(parents=True, exist_ok=True)
crash_files = list((VIDEO_ROOT / 'Crash-1500').glob('*.mp4'))
normal_files = list((VIDEO_ROOT / 'Normal').glob('*.mp4'))
if len(crash_files) != 1500 or len(normal_files) != 3000:
    assert ACCEPT_CCD_DATASET_TERMS, 'Review the CCD/BDD100K source terms, set ACCEPT_CCD_DATASET_TERMS=True, and rerun.'
    import gdown, zipfile
    archive_root = Path('/content/ccd_archives')  # ephemeral Colab disk, not laptop or Drive
    archive_root.mkdir(parents=True, exist_ok=True)
    sources = {}
    for name in ('Crash-1500.txt', 'Crash-1500.zip', 'Normal.zip'):
        existing = [p for p in DATA_ROOT.rglob(name) if p.is_file() and p.stat().st_size == EXPECTED_BYTES[name]]
        destination = VIDEO_ROOT / name if name.endswith('.txt') else archive_root / name
        if existing:
            sources[name] = existing[0]
        else:
            result = gdown.download(id=CCD_FILE_IDS[name], output=str(destination), quiet=False, resume=True)
            assert result and destination.stat().st_size == EXPECTED_BYTES[name], f'Incomplete official CCD file: {name}'
            sources[name] = destination
    if sources['Crash-1500.txt'] != VIDEO_ROOT / 'Crash-1500.txt':
        shutil.copy2(sources['Crash-1500.txt'], VIDEO_ROOT / 'Crash-1500.txt')
    for archive_name in ('Crash-1500.zip', 'Normal.zip'):
        with zipfile.ZipFile(sources[archive_name]) as archive:
            bad_member = archive.testzip()
            assert bad_member is None, f'Corrupt ZIP member: {bad_member}'
            archive.extractall(VIDEO_ROOT)
print('CCD root:', CCD_ROOT)

In [ ]:
import cv2
crash_files = sorted((CCD_ROOT / 'videos' / 'Crash-1500').glob('*.mp4'))
normal_files = sorted((CCD_ROOT / 'videos' / 'Normal').glob('*.mp4'))
annotation_path = CCD_ROOT / 'videos' / 'Crash-1500.txt'
print({'crash_videos': len(crash_files), 'normal_videos': len(normal_files), 'annotation_bytes': annotation_path.stat().st_size if annotation_path.exists() else 0})
assert len(crash_files) == 1500, 'CCD crash footage is incomplete.'
assert len(normal_files) == 3000, 'CCD normal footage is incomplete.'
assert annotation_path.stat().st_size == EXPECTED_BYTES['Crash-1500.txt']
assert all(p.stat().st_size > 0 for p in crash_files + normal_files), 'One or more video files are empty.'
for video_path in crash_files[:10] + normal_files[:10]:
    capture = cv2.VideoCapture(str(video_path)); ok, frame = capture.read(); frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT)); fps = capture.get(cv2.CAP_PROP_FPS); capture.release()
    assert ok and frame is not None and frame_count >= 50 and fps > 0, f'Unreadable or incomplete video: {video_path}'
print('Integrity sample passed for 10 crash and 10 normal videos.')

## Leakage-safe manifest
The split is deterministic by original source-video group. Accident timing comes only from official frame labels; normal clips receive no fabricated crash interval.

In [ ]:
MANIFEST = DATA_ROOT / 'ccd_manifest.csv'
subprocess.run([sys.executable, 'ml/training/prepare_manifests.py', '--source', 'ccd', '--root', str(CCD_ROOT), '--output', str(MANIFEST)], check=True)
import pandas as pd
manifest = pd.read_csv(MANIFEST)
display(manifest.groupby(['split', 'is_positive']).size().rename('videos'))
assert len(manifest) == 4500
assert manifest.video_id.is_unique
assert manifest.groupby('group_id').split.nunique().max() == 1, 'Group leakage detected'
assert manifest.media_path.map(lambda p: Path(p).is_file()).all(), 'Manifest references missing footage'

## Small real-data verification run
Before the full job, this balanced subset exercises real CCD decoding, YOLO tracking, all three splits, checkpointing, evaluation, and ONNX export. Its metrics are only a pipeline check, not the final model result.

In [ ]:
VERIFY_MANIFEST = DATA_ROOT / 'ccd_verify_manifest.csv'
verify_manifest = manifest.groupby(['split', 'is_positive'], group_keys=False).head(4).copy()
assert len(verify_manifest) == 24 and verify_manifest.groupby(['split', 'is_positive']).size().eq(4).all()
verify_manifest.to_csv(VERIFY_MANIFEST, index=False)
VERIFY_FEATURE_ROOT = DATA_ROOT / 'ccd_verify_motion_features'
subprocess.run([sys.executable, 'ml/training/extract_features.py', '--manifest', str(VERIFY_MANIFEST), '--output', str(VERIFY_FEATURE_ROOT), '--model', 'yolo11n.pt', '--sample-fps', '2', '--image-size', '320', '--confidence', '0.35'], check=True)
VERIFY_RUN_ROOT = DATA_ROOT / 'runs' / 'real_data_verification'
subprocess.run([sys.executable, 'ml/training/train_temporal.py', '--features', str(VERIFY_FEATURE_ROOT), '--output', str(VERIFY_RUN_ROOT), '--sequence-length', '4', '--stride', '2', '--epochs', '2', '--batch-size', '16', '--num-workers', '2', '--resume'], check=True)
print(json.dumps(json.loads((VERIFY_RUN_ROOT / 'evaluation.json').read_text()), indent=2))

## Production-compatible motion-feature extraction
Each completed video is an independent NPZ in Drive. Rerunning skips completed videos and resumes after interruption. A GPU is strongly recommended; without one, select Runtime → Change runtime type → GPU before this stage.

In [ ]:
FEATURE_ROOT = DATA_ROOT / 'ccd_motion_features'
extract_command = [sys.executable, 'ml/training/extract_features.py', '--manifest', str(MANIFEST), '--output', str(FEATURE_ROOT), '--model', 'yolo11n.pt', '--sample-fps', '2', '--image-size', '320', '--confidence', '0.35']
print('Extraction device:', 'GPU' if HAS_GPU else 'CPU (slow)')
subprocess.run(extract_command, check=True)

In [ ]:
feature_files = list(FEATURE_ROOT.glob('*.npz'))
report = json.loads((FEATURE_ROOT / 'extraction_report.json').read_text())
print({'feature_files': len(feature_files), 'failures': len(report['failures'])})
assert len(feature_files) == len(manifest), 'Extraction incomplete; rerun the preceding cell to resume.'
assert not report['failures'], report['failures'][:10]

## Train, validate, resume, and test
`last.pt` is updated every epoch in Drive. `--resume` restores model and optimizer state after interruption. Threshold selection uses validation only; the final report uses held-out test data.

In [ ]:
RUN_ROOT = DATA_ROOT / 'runs' / 'temporal_gru_v1'
train_command = [sys.executable, 'ml/training/train_temporal.py', '--features', str(FEATURE_ROOT), '--output', str(RUN_ROOT), '--sequence-length', '12', '--stride', '2', '--epochs', '30', '--batch-size', '256', '--num-workers', '2', '--resume']
subprocess.run(train_command, check=True)
evaluation = json.loads((RUN_ROOT / 'evaluation.json').read_text())
print(json.dumps(evaluation, indent=2))

## Held-out error review
Review the highest-scoring false positives against their source videos. Record whether they show braking, congestion, intersections, close proximity, tracking loss, or another pattern; these categories cannot be inferred honestly from CCD's temporal labels alone.

In [ ]:
predictions = pd.read_csv(RUN_ROOT / 'test_predictions.csv')
false_positives = predictions[(predictions.label == 0) & (predictions.model_prediction == 1)].sort_values('model_score', ascending=False)
print('Held-out false-positive windows:', len(false_positives))
display(false_positives.head(20))
false_positives.to_csv(RUN_ROOT / 'false_positives_for_review.csv', index=False)

## Verify PyTorch ↔ ONNX and prepare deployment
The exported score is supporting evidence, not proof of a crash. Keep `crash_temporal.onnx` beside its JSON sidecar.

In [ ]:
import numpy as np, onnxruntime as ort
from ml.training.train_temporal import TemporalGRU
meta = json.loads((RUN_ROOT / 'crash_temporal.json').read_text())
checkpoint = torch.load(RUN_ROOT / 'best.pt', map_location='cpu', weights_only=True)
model = TemporalGRU(); model.load_state_dict(checkpoint['model']); model.eval()
sample = np.zeros((1, meta['sequence_length'], len(meta['feature_names'])), dtype=np.float32)
with torch.no_grad(): torch_logit = model(torch.from_numpy(sample)).numpy()
session = ort.InferenceSession(str(RUN_ROOT / 'crash_temporal.onnx'), providers=['CPUExecutionProvider'])
onnx_logit = session.run(None, {session.get_inputs()[0].name: sample})[0]
np.testing.assert_allclose(torch_logit, onnx_logit, rtol=1e-5, atol=1e-5)
start = time.perf_counter()
for _ in range(200): session.run(None, {session.get_inputs()[0].name: sample})
print('ONNX CPU ms/inference:', round((time.perf_counter() - start) * 1000 / 200, 3))
print('Deploy both:', RUN_ROOT / 'crash_temporal.onnx', RUN_ROOT / 'crash_temporal.json')